# 01 · Config, schema, and seed

Load `configs/local.json`, create the MySQL `transactions` table from `transactions.csv`, and inspect the resulting schema.

This notebook stops at data prep. No model training yet.

In [ ]:
from cross_model_drift.config import load_config
from cross_model_drift.notebook import read_sql as _read_sql
from cross_model_drift.seed import (
    count_transactions,
    create_transactions_table,
    resolve_csv_path,
    seed_transactions,
)

config = load_config("local")
csv_path = resolve_csv_path(config)


def read_sql(sql: str):
    return _read_sql(sql, config)


config, csv_path

## CSV schema

Source file columns, as exported:

| column | meaning |
| --- | --- |
| `SENDER_ID` | sender key (12 hex chars) |
| `CREATED` | transaction timestamp |
| `PAYOUT_COUNTRY` | ISO-2 country |
| `PAYOUT_CURRENCY` | ISO-4217 currency |
| `AMOUNT_USD` | amount in USD |
| `FEE_USD` | fee in USD |
| `STATUS` | payout status |
| `ANTI_FRAUD_STATUS` | fraud label (`positive` / `negative`) — **target, not a feature** |
| `COMPLIANCE_STATUS` | separate downstream signal — **not used as a feature** |

In [ ]:
import pandas as pd

preview = pd.read_csv(csv_path, nrows=5)
preview

## Create table and load CSV

Uses MySQL `LOAD DATA LOCAL INFILE`. Re-run with `replace=True` to truncate and reload.

In [ ]:
create_transactions_table(config)
row_count = seed_transactions(config, csv_path)
print(f"rows in `{config.transactions_table}`: {row_count:,}")

## Explore MySQL schema

In [ ]:
read_sql(f"SHOW CREATE TABLE `{config.transactions_table}`")

In [ ]:
read_sql(f"DESCRIBE `{config.transactions_table}`")

In [ ]:
read_sql(
    f"""
    SELECT
        COUNT(*) AS n_rows,
        MIN(created) AS created_min,
        MAX(created) AS created_max,
        COUNT(DISTINCT sender_id) AS n_senders,
        COUNT(DISTINCT payout_country) AS n_countries,
        COUNT(DISTINCT payout_currency) AS n_currencies
    FROM `{config.transactions_table}`
    """
)

In [ ]:
read_sql(
    f"""
    SELECT status, COUNT(*) AS n
    FROM `{config.transactions_table}`
    GROUP BY status
    ORDER BY n DESC
    """
)

In [ ]:
read_sql(
    f"""
    SELECT anti_fraud_status, compliance_status, COUNT(*) AS n
    FROM `{config.transactions_table}`
    GROUP BY anti_fraud_status, compliance_status
    ORDER BY n DESC
    """
)

In [ ]:
read_sql(f"SELECT * FROM `{config.transactions_table}` ORDER BY created LIMIT 10")